In [1]:
import pandas as pd

df = pd.read_csv('./data/customer_travel.csv')

df.head()

,age,service,social,booked,target
0,34,6,0,1,0
1,34,5,1,0,1
2,37,3,1,0,0
3,30,2,0,0,0
4,30,1,0,0,0


In [2]:
pvt = len(df) // 2
df_a = df.iloc[:pvt]
df_b = df.iloc[pvt:]

df_a.shape

(400, 5)

In [3]:
from statsmodels.formula.api import logit

formula = "target ~ age + service + social + booked"

model = logit(formula, data=df_a).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.527521
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 target   No. Observations:                  400
Model:                          Logit   Df Residuals:                      395
Method:                           MLE   Df Model:                            4
Date:                Tue, 16 Jun 2026   Pseudo R-squ.:                 0.05254
Time:                        06:46:22   Log-Likelihood:                -211.01
converged:                       True   LL-Null:                       -222.71
Covariance Type:            nonrobust   LLR p-value:                 0.0001052
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.3314      1.204      1.937      0.053      -0.028       4.691
age           -0.1043      0.038     -2.781      0.005      -0.178      -0.031
service        0.0452      0.079      0.572      0.567      -0.110       0.200
social         0.1920      0.247      0.779      0.436      -0.291       0.675
booked        -0.9542      0.272     -3.512      0.000      -1.487      -0.422
==============================================================================
"""

In [4]:
model.pvalues[1:] >= 0.05 # 0.05 아래여야 유의
# 유의하지 않은 변수 service, social

age        False
service     True
social      True
booked     False
dtype: bool

In [5]:
formula_2 = "target ~ age + booked"

model_2 = logit(formula_2, data=df_a).fit()
model_2.summary()

# p값이 제일 큰

Optimization terminated successfully.
         Current function value: 0.528581
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 target   No. Observations:                  400
Model:                          Logit   Df Residuals:                      397
Method:                           MLE   Df Model:                            2
Date:                Tue, 16 Jun 2026   Pseudo R-squ.:                 0.05064
Time:                        06:46:22   Log-Likelihood:                -211.43
converged:                       True   LL-Null:                       -222.71
Covariance Type:            nonrobust   LLR p-value:                 1.265e-05
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.4581      1.184      2.076      0.038       0.137       4.779
age           -0.1025      0.037     -2.752      0.006      -0.176      -0.030
booked        -0.9461      0.267     -3.542      0.000      -1.470      -0.423
==============================================================================
"""

In [6]:
model_2.pvalues[1:].idxmax()

'age'

In [9]:
model_2.params[1:].abs().idxmax()

'booked'

In [10]:
model.llf

np.float64(-211.0084014929791)

In [12]:
-2 * model.llf

np.float64(422.0168029859582)

In [14]:
import numpy as np
np.exp(model.params['booked'] * 3)

np.float64(0.05711607448740832)

In [16]:
model.params[model.pvalues<0.05].sum()

np.float64(-1.058533826048773)

In [17]:
pred = model.predict(df_b)
pred = (pred > 0.5).astype(int)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(df_b['target'], pred)

In [18]:
accuracy

0.765

In [19]:
error_rate = 1 - accuracy
error_rate

0.235